In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent 
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

import torch
import torchaudio
from IPython.display import Audio, display
import librosa
import matplotlib.pyplot as plt
import numpy as np

# Import our processor
from processors.wrapper_encodec import EncodecProcessor

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Initialize 48kHz processor
sr = 48000
processor_48k_streamable = EncodecProcessor(sr=sr, streamable=True, )
processor_48k_chunks     = EncodecProcessor(sr=sr, streamable=False)

def full_autoencoding(audio_tensor, processor):
    sr = processor.sample_rate
    latent_cont_list, latent_cont_meta = processor.audio_to_latents(audio_tensor, sr)
    latent_cont_list = torch.cat(latent_cont_list, dim=-1)
    audio_from_latent_cont_48k = processor.decode_latents_audio(latent_cont_list, metadata=latent_cont_meta)
    return audio_from_latent_cont_48k    

# Multiscale FFT Loss
from auxiliar.losses import MultiscaleSpectrogramLoss
mss_loss = MultiscaleSpectrogramLoss(scales=[8192, 4096, 2048, 1024], 
                                    overlap=0.75)
spectral_loss_fn = lambda x,y: mss_loss(x.view(1,-1), y.view(1,-1))

def rms_imposer(audio, target_rms):
    current_rms = torch.sqrt(torch.mean(audio**2))
    scaling_factor = target_rms / (current_rms + 1e-8)  # Add small value to avoid division by zero
    return audio * scaling_factor

# Metadata for streaming mode can be fully faked
def metadata_maker(latent_content, scales = 1.0):
    metadata = {}
    # making padding_mask
    length = latent_content.shape[-1] * (48000//150)
    metadata['padding_mask'] = torch.ones((1, length), dtype=torch.int32, device=latent_content.device)
    # making audio_scales
    metadata['audio_scales'] = torch.tensor([[scales]], dtype=torch.float32, device=latent_content.device)
    return metadata

In [ ]:
def experiment(audio_path, length_seconds, segment_length=150, hop_size=30, overlap_factor_frames=10):
    audio_input, _ = librosa.load(audio_path, sr=sr, mono=True)
    audio_input = torch.tensor(audio_input).unsqueeze(0)  # Add batch dimension
    audio_input = audio_input[:, :int(sr*length_seconds)].to(processor_48k_streamable.device)  # Use only the first `length_seconds` seconds to speed up processing
    length_audio = audio_input.shape[-1]

    audio_full_autoencoded_stream = full_autoencoding(audio_input, processor_48k_streamable)
    audio_full_autoencoded_chunks = full_autoencoding(audio_input, processor_48k_chunks)
    # print(f"\nFull autoencoded audio shape: {audio_full_autoencoded.shape}")

    # Encode to latents
    latent_cont_list, latent_cont_meta = processor_48k_streamable.audio_to_latents(audio_input, 48000)

    # list of one tensor to tensor
    latent_cont_list = torch.cat(latent_cont_list, dim=-1)

    # Segmentation of the latent content (150 frames per second)
    segment_length   = segment_length                   # Number of latent frames per segment 
    audio_samples_per_segment = segment_length * (48000//150)  # Number of audio samples corresponding to each segment
    hop_size = hop_size                            # Number of latent frames to hop for the next segment 
    audio_samples_per_hop = hop_size * (48000//150)  # Number of audio samples corresponding to each hop
    length_of_latent_content = latent_cont_list.shape[-1]     # Total number of latent frames corresponding to the input audio duration
    # print(f"Length of latent content in frames: {length_of_latent_content}")
    overlap_factor_frames  = overlap_factor_frames
    overlap_factor_samples = overlap_factor_frames * (48000//150)

    # Create segments with hop size given (overlapping segments)
    segments = []
    for start in range(0, length_of_latent_content - segment_length + 1, hop_size):
        end = start + segment_length
        segment = latent_cont_list[:, :, start:end]  # Shape: (1, 128, segment_length)
        segments.append(segment)
    # print("number of segments:", len(segments))

    # Compute scales from real audio for each segment (using RMS as an example)
    scales          = []
    for start in range(0, audio_input.shape[-1] - audio_samples_per_segment+1, audio_samples_per_hop):
        end = start + audio_samples_per_hop
        segment = audio_input[:, start:end]  # Shape: (1, segment_length_in_samples)
        rms_value = torch.sqrt(torch.mean(segment**2))
        scales.append(rms_value.item())
    # print("number of scales:", len(scales))

    # Fake metadatas for each segment (using the computed scales)
    metadatas = []
    for i in range(len(segments)):
        metadata = metadata_maker(segments[i])
        # print(f"Metadata for segment {i+1}: ")
        # print("  Scales: ", metadata["audio_scales"])
        # print("  Padding mask: ", metadata["padding_mask"].shape)
        metadatas.append(metadata)

    # Decode from latents one by one
    audios = []
    for i in range(len(segments)):
        audio_from_segment = processor_48k_streamable.decode_latents_audio(segments[i], metadata=metadatas[i])
        audio_from_segment = rms_imposer(audio_from_segment, metadatas[i]["audio_scales"])
        audio_np = torch.mean(audio_from_segment[0], dim=0, keepdim=True).detach().cpu().numpy()
        # display(Audio(audio_np, rate=sr))
        audios.append(audio_from_segment)
    
    # Overlap and add the decoded segments to reconstruct the full audio
    reconstructed_audio = torch.zeros((1, 2, length_audio), device=audio_input.device)  # Assuming stereo output

    # First mask
    mask_hanning_window       = torch.hann_window(2*overlap_factor_samples, device=audios[0].device)  # Apply Hann window to reduce artifacts
    mask_right_hanning_window = mask_hanning_window[overlap_factor_samples:]  # Right half of the Hann window
    mask_left_ones            = torch.ones((audio_samples_per_segment-overlap_factor_samples), device=audios[0].device)
    mask_first                = torch.cat((mask_left_ones, mask_right_hanning_window), dim=0)
    time = (np.arange(mask_first.shape[0]) * 150 / 48000)  # x-axis in seconds
    plt.figure(figsize=(10, 4))
    plt.plot(time[:len(mask_left_ones)], mask_left_ones.cpu().numpy(), color='blue', label='Nothing to do here since it is the first segment ('+str(segment_length-overlap_factor_frames)+' frames)')
    plt.axvline(x=(len(mask_left_ones) * 150 / 48000), color='green', linestyle='--')
    plt.plot(time[len(mask_left_ones):], mask_right_hanning_window.cpu().numpy(), color='orange', label='Overlap with the next window ('+str(overlap_factor_frames)+' frames)')
    plt.title(f"First Mask (made of {segment_length} frames)")
    plt.xlabel("Time (frames)")
    plt.ylabel("Mask Value")
    plt.legend()
    max_time = int(np.ceil(time[-1]))
    steps = 15
    plt.xticks(np.arange(0, max_time+1, max_time//steps))
    plt.show()
    
    # Second mask
    mask_hanning_window = torch.hann_window(2*overlap_factor_samples, device=audios[0].device)  # Apply Hann window to reduce artifacts
    mask_left_hanning_window  = mask_hanning_window[:overlap_factor_samples]  # Left half of the Hann window
    mask_right_hanning_window = mask_hanning_window[overlap_factor_samples:]  # Right half of the Hann window
    mask_ones    = torch.ones((audio_samples_per_hop-overlap_factor_samples), device=audios[0].device)
    mask_zeros   = torch.zeros((audio_samples_per_segment - overlap_factor_samples - audio_samples_per_hop), device=audios[0].device)
    mask_general = torch.cat((mask_zeros, mask_left_hanning_window, mask_ones, mask_right_hanning_window), dim=0)
    time = (np.arange(mask_first.shape[0]) * 150 / 48000)  # x-axis in seconds
    plt.figure(figsize=(10, 4))
    idx = 0
    plt.plot(time[idx:idx+len(mask_zeros)], mask_zeros.cpu().numpy(), color='black', label='Get rid of the past ('+str(segment_length-overlap_factor_frames-hop_size)+' frames)')
    idx += len(mask_zeros)
    plt.axvline(x=(idx * 150 / 48000), color='green', linestyle='--')
    plt.plot(time[idx:idx+len(mask_left_hanning_window)], mask_left_hanning_window.cpu().numpy(), color='red', label='Overlap with the past window ('+str(overlap_factor_frames)+' frames)')
    idx += len(mask_left_hanning_window)
    plt.axvline(x=(idx * 150 / 48000), color='green', linestyle='--')
    plt.plot(time[idx:idx+len(mask_ones)], mask_ones.cpu().numpy(), color='blue', label='Ones ('+str(hop_size-overlap_factor_frames)+' frames)')
    idx += len(mask_ones)
    plt.axvline(x=(idx * 150 / 48000), color='green', linestyle='--')
    plt.plot(time[idx:idx+len(mask_right_hanning_window)], mask_right_hanning_window.cpu().numpy(), color='orange', label='Overlap with the next window ('+str(overlap_factor_frames)+' frames)')
    plt.title("General Mask for the other segments (made of "+str(segment_length)+" frames)")
    plt.xlabel("Time (frames)")
    plt.ylabel("Mask Value")
    plt.legend()
    max_time = int(np.ceil(time[-1]))
    steps = 15
    plt.xticks(np.arange(0, max_time+1, max_time//steps))
    plt.show()

    # First segment
    length_audio_segment = audios[0].shape[-1]
    reconstructed_audio[:, :, 0:length_audio_segment] += audios[0]*mask_first

    # Other segments
    for i in range(1, len(segments)):
        # print(f"Adding segment {i+1}/{len(segments)} to reconstruction")
        start = i * audio_samples_per_hop
        end = start + length_audio_segment
        audio_windowed = audios[i]*mask_general
        reconstructed_audio[:, :, start:end] += audio_windowed

    # Making all of them the same shape for loss computation and listening
    audio_input                   = audio_input[0]
    reconstructed_audio           = reconstructed_audio[0].mean(dim=0, keepdim=False)[:audio_input.shape[0]]
    audio_full_autoencoded_stream = audio_full_autoencoded_stream[0].mean(dim=0, keepdim=False)[:audio_input.shape[0]]
    audio_full_autoencoded_chunks = audio_full_autoencoded_chunks[0].mean(dim=0, keepdim=False)[:audio_input.shape[0]]

    # loss computation
    loss_full_autoencoding_stream = spectral_loss_fn(audio_input, audio_full_autoencoded_stream)
    loss_full_autoencoding_chunks = spectral_loss_fn(audio_input, audio_full_autoencoded_chunks)
    loss_segmented_autoencoding = spectral_loss_fn(audio_input, reconstructed_audio)

    # play!
    print(f"\nOriginal audio")
    display(Audio(audio_input.detach().cpu().numpy(), rate=sr))

    print(f"\n(loss = {loss_full_autoencoding_stream.item():.4f}) Full autoencoded audio from streamable encodec (all at once)")
    display(Audio(audio_full_autoencoded_stream.detach().cpu().numpy(), rate=sr))

    print(f"\n(loss = {loss_full_autoencoding_chunks.item():.4f}) Full autoencoded audio non-streamable encodec (using default chunks processing of 1 second with overlap of 10ms)")
    display(Audio(audio_full_autoencoded_chunks.detach().cpu().numpy(), rate=sr))
    
    print(f"\n(loss = {loss_segmented_autoencoding.item():.4f}) Full encoded with streamable encodec and then decoded in segments with the same model (encoding/decoding size discrepancy here). Decoded segments are of size {segment_length} but only the last {hop_size}+{overlap_factor_frames} frames are added to the reconstruction to reduce artifacts, using overlap and add with Hann windows.")
    display(Audio(reconstructed_audio.detach().cpu().numpy(), rate=sr))

In [ ]:
experiment(audio_path = "../samples/guitar.wav", length_seconds = 10, segment_length=450, hop_size=150, overlap_factor_frames=10)